# Inter-Annotator Agreement Analysis for STREAMS Evaluations

This notebook provides comprehensive analysis and visualization of inter-annotator agreement for STREAMS (Standards for Technical Reporting in Environmental and host-Associated Microbiome Studies) evaluations.

## Overview

- **Purpose**: Analyze agreement between multiple reviewers evaluating research papers against STREAMS criteria
- **Metrics**: Cohen's Kappa, simple agreement rates, completion rates
- **Data**: 140+ human-reviewed STREAMS evaluations from `evals/streams-gdrive/`

## Setup

In [1]:
import sys
from pathlib import Path
import warnings

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Import core modules
from streams_agent.evals.inter_annotator_agreement import compute_inter_annotator_agreement
from streams_agent.evals.visualization import (
    plot_kappa_distribution,
    plot_kappa_by_paper,
    plot_agreement_matrix_interactive,
    create_agreement_summary_plot,
    plot_completion_rates
)
from streams_agent.loaders import load_streams_assessment_from_excel, load_streams_checklist_from_csv
from streams_agent.models import AssessmentRating

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✅ Setup complete!")

✅ Setup complete!


## Data Loading and Basic Statistics

First, let's load the evaluation data and compute basic statistics.

In [2]:
# Define paths
eval_dir = project_root / "evals" / "streams-gdrive" / "Benchmarking STREAMS Exemplars"
template_path = project_root / "src" / "streams_agent" / "template" / "streams.csv"

print(f"📁 Evaluation directory: {eval_dir}")
print(f"📄 Template path: {template_path}")
print(f"📊 Directory exists: {eval_dir.exists()}")
print(f"📋 Template exists: {template_path.exists()}")

if eval_dir.exists():
    # Count evaluation files
    eval_files = list(eval_dir.glob("*.xlsx"))
    eval_files = [f for f in eval_files if not f.name.startswith('~')]  # Skip temp files
    print(f"\n📈 Found {len(eval_files)} evaluation files")
    
    # Load STREAMS checklist template
    if template_path.exists():
        checklist = load_streams_checklist_from_csv(template_path)
        print(f"📋 STREAMS checklist: {checklist.name} v{checklist.version}")
        print(f"📝 Total checklist items: {len(checklist.items)}")
    else:
        print("⚠️  Template file not found")
        checklist = None
else:
    print("❌ Evaluation directory not found")
    eval_files = []
    checklist = None

📁 Evaluation directory: /Users/cjm/repos/evals/streams-gdrive/Benchmarking STREAMS Exemplars
📄 Template path: /Users/cjm/repos/src/streams_agent/template/streams.csv
📊 Directory exists: False
📋 Template exists: False
❌ Evaluation directory not found


## Overall Inter-Annotator Agreement Analysis

Compute Cohen's Kappa and agreement rates across all papers with multiple reviewers.

In [3]:
if eval_dir.exists():
    # Compute inter-annotator agreement
    print("🔄 Computing inter-annotator agreement...")
    results = compute_inter_annotator_agreement(eval_dir, min_reviewers=2)
    
    print("\n" + "="*50)
    print("📊 INTER-ANNOTATOR AGREEMENT RESULTS")
    print("="*50)
    
    print(f"🎯 Overall Cohen's Kappa: {results['overall_kappa']:.3f}")
    print(f"📋 Simple Agreement Rate: {results['agreement_rate']:.1%}")
    print(f"📄 Papers Analyzed: {results['num_papers_analyzed']}")
    print(f"🔄 Pairwise Comparisons: {results['num_pairwise_comparisons']}")
    
    # Interpret Kappa score
    kappa = results['overall_kappa']
    if kappa < 0:
        interpretation = "Poor (worse than chance)"
    elif kappa < 0.2:
        interpretation = "Slight"
    elif kappa < 0.4:
        interpretation = "Fair"
    elif kappa < 0.6:
        interpretation = "Moderate"
    elif kappa < 0.8:
        interpretation = "Substantial"
    else:
        interpretation = "Almost Perfect"
    
    print(f"\n🎭 Kappa Interpretation: {interpretation}")
    print("\n📖 Kappa Scale (Landis & Koch, 1977):")
    print("   < 0.00: Poor")
    print("   0.00-0.20: Slight")
    print("   0.21-0.40: Fair")
    print("   0.41-0.60: Moderate")
    print("   0.61-0.80: Substantial")
    print("   0.81-1.00: Almost Perfect")
else:
    results = None
    print("❌ Cannot compute agreement - evaluation directory not found")

❌ Cannot compute agreement - evaluation directory not found


## Interactive Summary Dashboard

A comprehensive overview of all agreement metrics in an interactive dashboard.

In [4]:
if results:
    # Create interactive summary dashboard
    summary_fig = create_agreement_summary_plot(results)
    summary_fig.show()
else:
    print("❌ No results available for dashboard")

❌ No results available for dashboard


## Distribution of Kappa Scores

Analyze how agreement varies across different papers.

In [5]:
if results:
    # Plot kappa distribution
    kappa_dist_fig = plot_kappa_distribution(results)
    plt.show()
    
    # Show statistics
    per_paper_kappa = results.get('per_paper_kappa', {})
    if per_paper_kappa and isinstance(per_paper_kappa, dict):
        kappa_values = list(per_paper_kappa.values())
        kappa_df = pd.DataFrame({'kappa': kappa_values})
        
        print("\n📊 Kappa Score Statistics:")
        print(kappa_df.describe())
        
        # Count papers by agreement level
        agreement_levels = {
            'Poor (< 0)': sum(1 for k in kappa_values if k < 0),
            'Slight (0-0.2)': sum(1 for k in kappa_values if 0 <= k < 0.2),
            'Fair (0.2-0.4)': sum(1 for k in kappa_values if 0.2 <= k < 0.4),
            'Moderate (0.4-0.6)': sum(1 for k in kappa_values if 0.4 <= k < 0.6),
            'Substantial (0.6-0.8)': sum(1 for k in kappa_values if 0.6 <= k < 0.8),
            'Almost Perfect (0.8+)': sum(1 for k in kappa_values if k >= 0.8)
        }
        
        print("\n🎭 Papers by Agreement Level:")
        for level, count in agreement_levels.items():
            if count > 0:
                print(f"   {level}: {count} papers")
else:
    print("❌ No results available for kappa distribution plot")

❌ No results available for kappa distribution plot


## Individual Paper Analysis

Examine agreement scores for individual papers to identify best and worst cases.

In [6]:
if results:
    # Plot kappa scores by paper (top 20)
    paper_kappa_fig = plot_kappa_by_paper(results, top_n=20)
    plt.show()
    
    # Show best and worst performing papers
    per_paper_kappa = results.get('per_paper_kappa', {})
    if per_paper_kappa and isinstance(per_paper_kappa, dict):
        sorted_papers = sorted(per_paper_kappa.items(), key=lambda x: x[1], reverse=True)
        
        print("\n🏆 Top 5 Papers (Highest Agreement):")
        for i, (paper_id, kappa) in enumerate(sorted_papers[:5], 1):
            print(f"   {i}. {paper_id}: κ = {kappa:.3f}")
        
        print("\n⚠️  Bottom 5 Papers (Lowest Agreement):")
        for i, (paper_id, kappa) in enumerate(sorted_papers[-5:], 1):
            print(f"   {i}. {paper_id}: κ = {kappa:.3f}")
else:
    print("❌ No results available for individual paper analysis")

❌ No results available for individual paper analysis


## Completion Rate Analysis

Analyze how completely evaluators filled out the STREAMS checklist.

In [7]:
if eval_dir.exists():
    # Plot completion rates
    completion_fig = plot_completion_rates(eval_dir, min_assessments=2)
    plt.show()
    
    # Calculate overall completion statistics
    all_assessments = []
    for file_path in eval_dir.glob("*.xlsx"):
        if file_path.name.startswith('~'):
            continue
        assessment = load_streams_assessment_from_excel(file_path)
        if assessment:
            all_assessments.append(assessment)
    
    if all_assessments:
        completion_rates = [a.get_completion_rate() for a in all_assessments]
        avg_completion = sum(completion_rates) / len(completion_rates)
        
        print(f"\n📊 Completion Rate Statistics:")
        print(f"   Total assessments: {len(all_assessments)}")
        print(f"   Average completion rate: {avg_completion:.1%}")
        print(f"   Min completion rate: {min(completion_rates):.1%}")
        print(f"   Max completion rate: {max(completion_rates):.1%}")
        
        # Count fully completed assessments
        fully_completed = sum(1 for rate in completion_rates if rate >= 0.95)
        print(f"   Fully completed (≥95%): {fully_completed} ({fully_completed/len(completion_rates):.1%})")
else:
    print("❌ Cannot analyze completion rates - evaluation directory not found")

❌ Cannot analyze completion rates - evaluation directory not found


## Deep Dive: Specific Paper Analysis

Choose a specific paper to examine detailed agreement patterns between reviewers.

In [8]:
# Select a paper for detailed analysis
if results and results.get('per_paper_kappa'):
    # Choose the paper with highest agreement for demo
    per_paper_kappa = results['per_paper_kappa']
    if isinstance(per_paper_kappa, dict) and per_paper_kappa:
        best_paper = max(per_paper_kappa.items(), key=lambda x: x[1])
        selected_paper_id = best_paper[0]
        
        print(f"🎯 Analyzing paper: {selected_paper_id} (κ = {best_paper[1]:.3f})")
        
        # Create interactive agreement matrix
        if eval_dir.exists():
            interactive_fig = plot_agreement_matrix_interactive(eval_dir, selected_paper_id)
            if interactive_fig:
                interactive_fig.show()
            else:
                print("⚠️  Could not create interactive plot for this paper")
        
        # Load assessments for this paper to show details
        paper_assessments = []
        for file_path in eval_dir.glob("*.xlsx"):
            if file_path.name.startswith('~'):
                continue
            assessment = load_streams_assessment_from_excel(file_path)
            if assessment and assessment.paper_id == selected_paper_id:
                paper_assessments.append(assessment)
        
        if paper_assessments:
            print(f"\n📋 Found {len(paper_assessments)} assessments for {selected_paper_id}:")
            for assessment in paper_assessments:
                completion_rate = assessment.get_completion_rate()
                rated_items = sum(1 for a in assessment.assessments if a.rating is not None)
                print(f"   {assessment.evaluator_id}: {completion_rate:.1%} complete ({rated_items}/{len(assessment.assessments)} items)")
    else:
        print("❌ No per-paper kappa data available")
else:
    print("❌ No results available for detailed paper analysis")

❌ No results available for detailed paper analysis


## Rating Distribution Analysis

Examine how different rating types (Yes/No/NA) are distributed across evaluations.

In [9]:
if eval_dir.exists():
    # Collect all ratings
    all_ratings = []
    for file_path in eval_dir.glob("*.xlsx"):
        if file_path.name.startswith('~'):
            continue
        assessment = load_streams_assessment_from_excel(file_path)
        if assessment:
            for item_assessment in assessment.assessments:
                if item_assessment.rating:
                    all_ratings.append(item_assessment.rating.value)
    
    if all_ratings:
        # Create rating distribution plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Count plot
        rating_counts = pd.Series(all_ratings).value_counts()
        colors = ['lightgreen', 'lightcoral', 'lightblue', 'lightyellow']
        rating_counts.plot(kind='bar', ax=ax1, color=colors[:len(rating_counts)])
        ax1.set_title('Distribution of Ratings (Counts)')
        ax1.set_xlabel('Rating')
        ax1.set_ylabel('Count')
        ax1.tick_params(axis='x', rotation=45)
        
        # Add count labels on bars
        for i, (rating, count) in enumerate(rating_counts.items()):
            ax1.text(i, count + max(rating_counts) * 0.01, str(count), 
                    ha='center', va='bottom', fontweight='bold')
        
        # Pie chart
        ax2.pie(rating_counts.values, labels=rating_counts.index, autopct='%1.1f%%',
               colors=colors[:len(rating_counts)], startangle=90)
        ax2.set_title('Distribution of Ratings (Proportions)')
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        total_ratings = len(all_ratings)
        print(f"\n📊 Rating Distribution Statistics:")
        print(f"   Total rated items: {total_ratings:,}")
        for rating, count in rating_counts.items():
            percentage = count / total_ratings * 100
            print(f"   {rating}: {count:,} ({percentage:.1f}%)")
        
        # Calculate agreement tendency
        yes_rate = rating_counts.get('Yes', 0) / total_ratings
        no_rate = rating_counts.get('No', 0) / total_ratings
        na_rate = rating_counts.get('NA', 0) / total_ratings
        
        print(f"\n🎯 Key Insights:")
        print(f"   Positive compliance rate: {yes_rate:.1%}")
        print(f"   Non-compliance rate: {no_rate:.1%}")
        print(f"   Not applicable rate: {na_rate:.1%}")
        
        if yes_rate > 0.6:
            print(f"   📈 High overall compliance with STREAMS guidelines")
        elif yes_rate < 0.4:
            print(f"   📉 Low overall compliance with STREAMS guidelines")
        else:
            print(f"   📊 Moderate overall compliance with STREAMS guidelines")
else:
    print("❌ Cannot analyze rating distribution - evaluation directory not found")

❌ Cannot analyze rating distribution - evaluation directory not found


## Conclusions and Recommendations

Summary of findings and suggestions for improving evaluation consistency.

In [10]:
if results:
    print("📝 ANALYSIS SUMMARY AND RECOMMENDATIONS")
    print("="*50)
    
    kappa = results['overall_kappa']
    agreement_rate = results['agreement_rate']
    num_papers = results['num_papers_analyzed']
    
    print(f"\n🎯 Overall Assessment:")
    print(f"   Cohen's Kappa: {kappa:.3f} ({interpretation})")
    print(f"   Simple Agreement: {agreement_rate:.1%}")
    print(f"   Papers Analyzed: {num_papers}")
    
    print(f"\n💡 Recommendations:")
    
    if kappa < 0.4:
        print("   🔴 LOW AGREEMENT - Consider:")
        print("     • Additional evaluator training")
        print("     • Clearer STREAMS guidelines")
        print("     • Pilot evaluation sessions")
        print("     • Regular calibration meetings")
    elif kappa < 0.6:
        print("   🟡 MODERATE AGREEMENT - Consider:")
        print("     • Periodic evaluator calibration")
        print("     • Review of ambiguous items")
        print("     • Additional examples for unclear criteria")
    else:
        print("   🟢 GOOD AGREEMENT - Maintain:")
        print("     • Current training procedures")
        print("     • Regular quality checks")
        print("     • Ongoing evaluator support")
    
    # Completion rate recommendations
    if 'avg_completion' in locals():
        if avg_completion < 0.8:
            print("\n📋 COMPLETION RATES - Consider:")
            print("     • Simplifying evaluation interface")
            print("     • Reducing evaluation burden")
            print("     • Providing completion incentives")
    
    print(f"\n🔍 Areas for Further Investigation:")
    print("   • Item-level agreement patterns")
    print("   • Evaluator-specific biases")
    print("   • Impact of paper characteristics on agreement")
    print("   • Training effectiveness measurement")
    
    print(f"\n📊 Data Quality Notes:")
    print(f"   • Based on {results['num_pairwise_comparisons']} pairwise comparisons")
    print(f"   • Minimum 2 reviewers per paper")
    print(f"   • Excludes papers with <5 common rated items")

print("\n" + "="*50)
print("📋 Analysis complete! Use the visualizations above to explore")
print("specific patterns and identify areas for improvement.")
print("="*50)


📋 Analysis complete! Use the visualizations above to explore
specific patterns and identify areas for improvement.


## Export Results

Save key results and visualizations for reporting.

In [11]:
# Create output directory
output_dir = project_root / "output" / "iaa_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

if results:
    # Save summary statistics
    import json
    
    summary_stats = {
        'overall_kappa': results['overall_kappa'],
        'agreement_rate': results['agreement_rate'],
        'num_papers_analyzed': results['num_papers_analyzed'],
        'num_pairwise_comparisons': results['num_pairwise_comparisons'],
        'interpretation': interpretation
    }
    
    with open(output_dir / 'iaa_summary.json', 'w') as f:
        json.dump(summary_stats, f, indent=2)
    
    # Save per-paper results
    per_paper_kappa = results.get('per_paper_kappa', {})
    if per_paper_kappa and isinstance(per_paper_kappa, dict):
        per_paper_df = pd.DataFrame([
            {'paper_id': paper_id, 'kappa': kappa}
            for paper_id, kappa in per_paper_kappa.items()
        ])
        per_paper_df.to_csv(output_dir / 'per_paper_kappa.csv', index=False)
    
    print(f"✅ Results exported to: {output_dir}")
    print(f"   📄 Summary: iaa_summary.json")
    print(f"   📊 Per-paper data: per_paper_kappa.csv")
else:
    print("❌ No results to export")

❌ No results to export
